# 第41章 误差线与区间图（errorbar）

<!-- module-learning-arc:start -->
> **Matplotlib 模块主线｜第 10 / 12 步：表达不确定性与多视角证据**
>
> **持续应用背景：** 制作经营周会一页报告：把趋势、比较、分布和异常证据组织成有主次、可直接用于会议的静态页面。
>
> **承接上一阶段：** 饼图与环形图（pie）  →  **本章任务：** 误差线与区间图（errorbar）  →  **下一步：** 子图与组合图（subplots）
>
> **大作业连接：** 本章练习将成为《经营周会一页报告》的一部分，最终需要从周会问题出发选择互补图形，完成视觉层级、注释审阅与独立导出。
<!-- module-learning-arc:end -->


## 本章场景

在真实业务数据里，一个数字往往不是孤零零的“点”，而是带着波动和不确定性——这个月的平均销售额到底准不准，不同方案的预测区间有多宽，测量结果会不会忽高忽低。


## 本章目标

学完本章，你将能够：

- **理解**：理解「误差线与区间图（errorbar）」的适用场景、数据结构要求，以及它想帮你读出的规律。
- **操作**：能按参数用相应绘图接口画出「误差线与区间图（errorbar）」，并做必要的美化、注释与导出。
- **迁移**：能换一份真实经营数据，独立画出同类型的「误差线与区间图（errorbar）」并读出其中的结论。


## 适用场景

**背景引入**：在真实业务数据里，一个数字往往不是孤零零的“点”，而是带着波动和不确定性——这个月的平均销售额到底准不准，不同方案的预测区间有多宽，测量结果会不会忽高忽低。误差线与区间图正是用来把这些“不确定性”画出来的图表：它用中心点和棒线（或带状区间）同时展示“大概是多少”以及“波动有多大”。学会它，你就能从报表里判断数据是稳还是浮动，避免把一个偶然的波动误当成真实的差异。 打个比方：误差线就像射击靶上的弹着点——中心点是平均水平，棒线是每次结果的波动范围；棒越长，说明结果越不稳定，越不能只拿一次高分当真，它帮你判断一个数字是「稳」还是「飘」。

展示均值及标准差、置信区间、预测区间或测量误差。


## 数据结构

中心估计值及对应的对称或非对称误差。


## 本章练习任务

运行基础图表后，完成以下任务：

1. 将 capsize 参数从 4 改为 8，观察端帽长度对误差线可读性的影响
2. 调整 elinewidth 参数（如 1.5 或 3），说明误差线粗细的视觉效果
3. 修改 fill_between 的 alpha 值（如 0.1 或 0.35），对比不同透明度的区间显示


## 图表与参数速查

先用这张表建立本章的方法地图；每一行后面都有对应的独立示例或练习。

| 类别 | 常用方法或写法 | 主要用途 | 需要特别注意 |
| --- | --- | --- | --- |
| 基础图表 | `np.array()`、`plt.subplots()`、`ax.errorbar()`、`ax.set()` | 展示均值及标准差、置信区间、预测区间或测量误差。 | 不说明误差类型 |
| 进阶变体 | `np.array()`、`plt.subplots()`、`ax.plot()`、`ax.fill_between()` | 在基础图表上增加分组、注释、布局或交互 | 误差线过粗遮挡中心值 |
| 关键参数 | `yerr/xerr` | 误差 | 不说明误差类型 |
| 关键参数 | `capsize` | 端帽 | 误差线过粗遮挡中心值 |
| 关键参数 | `elinewidth` | 误差线宽 | 把预测区间解释为置信区间 |
| 关键参数 | `fill_between` | 连续区间 | 不说明误差类型 |


## 准备可复现数据

先完成导入和数据准备，后续单元格只负责一种图表或一种分析动作。


<!-- math-foundation:chapter-41 -->
### 数学推导｜均值的不确定性区间

> 阅读方法：先跟着步骤理解每个量怎样产生，再看最后的可计算形式；不需要脱离业务场景死记公式。

**第 1 步｜样本均值存在抽样波动。** 独立同分布条件下 $\operatorname{Var}(\bar X)=\sigma^2/n$。

**第 2 步｜用样本标准差估计未知的 $\sigma$。** 得到 $SE\approx s/\sqrt n$。

**第 3 步｜用标准化分布给出区间。** 大样本近似下

$$
\frac{\bar X-\mu}{SE}\approx N(0,1)
$$

标准正态中约 95% 落在 $[-1.96,1.96]$，移项后得到 $\bar x\pm1.96SE$。小样本时应把 1.96 换成相应的 $t$ 分位数。

**把上面的关系收束为本章计算式：**

$$
CI_{95\%}\approx \bar{x}\pm1.96\frac{s}{\sqrt{n}}
$$

**符号解释：** $\bar{x}$ 是样本均值，$s$ 是样本标准差，$n$ 是样本量。

**代码对应：** 统计图中的误差线应明确表示标准差、标准误还是置信区间。

**使用边界：** 该近似依赖样本与分布条件；小样本或偏态数据可考虑 bootstrap。


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# 中文字体支持：由平台运行时自动配置
# 说明：Matplotlib 默认字体不含中文字形，中文会显示成方框。
#      本平台在运行每个绘图 cell 前，会自动注册可用的中文字体并设置
#      font.sans-serif / axes.unicode_minus，因此这里不需要手动 import
#      或 addfont，直接使用即可。

# 1️⃣ 数据导入：读取订单数据（指定列类型降低内存、加快分组）
transactions = pd.read_csv(
    "/datasets/uci_online_retail_200k.csv",
    parse_dates=["InvoiceDate"],
    dtype={"Country": "category"},
)
print(f"数据规模：{len(transactions):,} 行 × {transactions.shape[1]} 列")


In [ ]:
# 2️⃣ 特征工程：构造分析所需字段与聚合结果
transactions["amount"] = transactions["Quantity"] * transactions["UnitPrice"]

# 有效订单：数量与单价均为正（退货/取消行不参与月度统计）
completed = transactions.query("Quantity > 0 and UnitPrice > 0")
completed["month"] = (
    completed["InvoiceDate"].dt.to_period("M").astype("string")
)

# 月度聚合：销售额（元）与订单数
monthly_summary = completed.groupby("month").agg(
    sales=("amount", "sum"), orders=("InvoiceNo", "nunique")
)
months = monthly_summary.index.to_numpy()[:6]  # 本章取前 6 个月
sales = (monthly_summary["sales"] / 10_000).to_numpy()[:6]
orders = monthly_summary["orders"].to_numpy()[:6]
profit = sales * 0.18  # 简化假设：利润约为销售额的 18%

# 区域构成：销售额前 4 国，统计「销售 vs 退货」两部分（单位：万元）
top = completed.groupby("Country")["amount"].sum().nlargest(4).index
rows = transactions[transactions["Country"].isin(top)].copy()
rows["flow"] = np.where(rows["Quantity"] > 0, "销售", "退货")
regional = (
    pd.crosstab(
        rows["Country"].astype(str),
        rows["flow"],
        values=rows["amount"].abs(),
        aggfunc="sum",
    )
    / 10_000
).fillna(0)
regions = regional.index.to_numpy()
online = regional["销售"].to_numpy()
offline = regional["退货"].to_numpy()

# 固定随机种子抽样 2000 条，供分布图使用，保证每次运行结果一致
samples = completed["amount"].sample(2_000, random_state=25).to_numpy()
print(f"有效订单：{len(completed):,} 行")


## 例 1｜最小可用图表

先保留必要的编码：位置、颜色或大小。图表标题、坐标轴和单位应能让读者脱离代码理解结果。


In [ ]:
import matplotlib.pyplot as plt

average = np.array([120, 148, 139, 176, 205, 228])
standard_error = np.array([6, 8, 7, 9, 11, 10])
fig, ax = plt.subplots(figsize=(8, 4.2))
ax.errorbar(
    months,
    average,
    yerr=standard_error,
    marker="o",
    capsize=4,
    color="#1a73e8",
    ecolor="#8ab4f8",
)
ax.set(title="月度销售额及标准误", ylabel="销售额（万元）")
fig.tight_layout()
plt.show()


In [ ]:
# （自动维护）练习上下文快照 1：参考答案将基于此刻的变量运行
_pds_snap_1 = dict(globals())


**练一练**：在上面的误差线图中，把端帽长度 `capsize` 从 4 改为 8，重新运行并观察误差线两端的帽盖发生了什么变化。

提示：端帽（capsize）是误差线两端的小横线，数值越大帽盖越长、越醒目。请在下方代码框中填写完整代码，然后运行 `# 自检` 部分确认代码正确。


In [ ]:
try:
    pass
    # 请在下方填写代码：把误差线端帽长度 capsize 从 4 改为 8，重画误差线图。

except Exception as _pds_err:
    print("（练习尚未完成或未填全：", _pds_err, "）")


## 例 2｜进阶变体

在基础图表可读的前提下增加分组、布局、注释或交互。新增编码必须服务于一个明确问题。


In [ ]:
import matplotlib.pyplot as plt

forecast = np.array([130, 145, 162, 181, 198, 216])
lower = forecast - np.array([12, 13, 14, 16, 18, 20])
upper = forecast + np.array([12, 13, 14, 16, 18, 20])
fig, ax = plt.subplots(figsize=(8, 4.2))
ax.plot(months, forecast, marker="o", color="#188038", label="预测")
ax.fill_between(
    months, lower, upper, color="#188038", alpha=0.18, label="预测区间"
)
ax.set(title="销售预测及不确定区间", ylabel="销售额（万元）")
ax.legend(frameon=False)
fig.tight_layout()
plt.show()


## 参数说明

- yerr/xerr：误差
- capsize：端帽
- elinewidth：误差线宽
- fill_between：连续区间


## 结果解读

先明确区间代表标准差、标准误还是置信区间；重叠不等于没有差异。


## 本章实训：图表只改一个编码

这一组实验专门训练“观察一个结果 → 只改一个变量 → 解释变化”。先运行第一个代码单元格，再运行第二个。


In [ ]:
import matplotlib.pyplot as plt

_demo_months = ["1月", "2月", "3月", "4月"]
_demo_sales = [120, 150, 138, 190]
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(_demo_months, _demo_sales, marker="o")
ax.set_title("月度销售额")
ax.set_ylabel("销售额（万元）")
ax.grid(alpha=0.25)
plt.show()


### 第一个结果怎么读

标题、坐标轴和单位让读者知道图表回答什么问题。没有这些文字，图形即使画出来也不完整。

请记录：输入是什么、输出是什么、输出支持了哪一个结论。


In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.bar(months, sales, color="#2563EB")
ax.axhline(
    sum(sales) / len(sales), color="#DC2626", linestyle="--", label="平均值"
)
ax.set_title("月度销售额与平均值")
ax.set_ylabel("销售额（万元）")
ax.legend()
plt.show()


### 第二个结果怎么读

第二个实验把折线改成柱状图，并增加平均线。请说明：哪种图更适合看趋势，哪种图更适合比较单月差异？

迁移任务：把一个输入值、一个字段或一个图表参数换成自己的例子，再用一句话解释变化。


## 错误恢复：图表能画出但读不懂怎么办

真实数据和真实代码都会出问题。本节先观察问题，再用一个明确的检查或修复步骤恢复运行。


In [ ]:
import matplotlib.pyplot as plt

_demo_months = ["1月", "2月", "3月"]
_demo_sales = [120, 150, 138]
fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(_demo_months, _demo_sales, marker="o")
ax.set_title("月度销售额")
ax.set_xlabel("月份")
ax.set_ylabel("销售额（万元）")
ax.grid(alpha=0.25)
plt.show()


### 错误恢复步骤

1. 先看错误类型、字段或数据形状。
2. 判断问题发生在输入、处理中间结果还是输出。
3. 修复后重新检查结果，而不是只让代码不报错。

图形没有报错不等于结果可用。遇到“看不懂”的图，优先补标题、坐标轴、单位和关键参照线。

迁移任务：把示例中的输入换成一组会触发问题的数据，并记录你的修复规则。


## 易错点提醒

- 不说明误差类型
- 误差线过粗遮挡中心值
- 把预测区间解释为置信区间


## 练习与作业

请使用同一份数据完成下面任务，并说明你选择该图表的原因。完成后补充：图表回答了什么问题、最重要的视觉信号是什么、还有哪些信息无法从图中得出。


## 独立迁移练习

复制最接近的示例，只修改一种视觉编码，并说明阅读任务如何变化。

先在下面单元格完成自己的版本；需要参考时再回看紧邻的示例或参考实现。


In [ ]:
# （自动维护）练习上下文快照 2：参考答案将基于此刻的变量运行
_pds_snap_2 = dict(globals())


In [ ]:
try:
    # 独立迁移练习：用填充区间替代误差棒，观察另一种不确定性表达
    # 【目标】同一份不确定性，可以用误差棒或带状区间两种方式呈现，练习对比。
    import matplotlib.pyplot as plt

    # 起点示例(已可运行)：用 fill_between 画出均值±标准误的带状区间。
    average = np.array([120, 148, 139, 176, 205, 228])
    standard_error = np.array([6, 8, 7, 9, 11, 10])
    fig, ax = plt.subplots(figsize=(8, 4.2))
    ax.plot(months, average, marker="o", color="#1a73e8")
    ax.fill_between(
        months,
        average - standard_error,
        average + standard_error,
        color="#1a73e8",
        alpha=0.2,
    )
    ax.set(title="月度销售额及标准误区间", ylabel="销售额（万元）")
    fig.tight_layout()
    plt.show()

    # ---- 反思记录：误差棒 vs 带状区间，视觉传达有何不同 ----
    change_note = "待填写"
    expected_change = "待填写"
    observed_change = "运行后填写"
    print(f"改动：{change_note}")
    print(f"预期：{expected_change}")
    print(f"观察：{observed_change}")

except Exception as _pds_err:
    print("（练习尚未完成或未填全：", _pds_err, "）")


## 小结

用误差线和带状区间表达估计值的不确定性、波动范围或上下界。


### 你已经掌握

- 判断误差线与区间图（errorbar）的适用场景
- 准备与图表匹配的数据结构
- 从基础图表扩展到分组、注释或交互变体
- 按照业务问题解读图表并说明结论边界


### 关键参数

| 参数 | 作用 |
| --- | --- |
| `yerr/xerr` | 误差 |
| `capsize` | 端帽 |
| `elinewidth` | 误差线宽 |
| `fill_between` | 连续区间 |


### 需要注意

- 不说明误差类型
- 误差线过粗遮挡中心值
- 把预测区间解释为置信区间


## 参考答案


### 本章练习


In [ ]:
# 恢复练习 1 时的变量上下文（后面的示例覆盖过这些名字）
globals().update(_pds_snap_1)


In [ ]:
# ===== 参考答案 =====
# capsize 控制误差线两端的端帽（帽盖）长度，数值越大帽盖越长、越醒目，
# 越容易被读者辨认出误差范围。
fig, ax = plt.subplots(figsize=(8, 4.2))
eb = ax.errorbar(
    months,
    average,
    yerr=standard_error,
    marker="o",
    capsize=8,
    color="#1a73e8",
    ecolor="#8ab4f8",
)
ax.set(title="月度销售额及标准误（端帽加长）", ylabel="销售额（万元）")
fig.tight_layout()


### 本章练习


In [ ]:
# 恢复练习 2 时的变量上下文（后面的示例覆盖过这些名字）
globals().update(_pds_snap_2)


In [ ]:
import matplotlib.pyplot as plt

means = np.array([4.2, 3.8, 4.5, 4.0])
errors = np.array([0.18, 0.22, 0.16, 0.20])
fig, ax = plt.subplots(figsize=(7.5, 4.2))
ax.bar(
    regions,
    means,
    yerr=errors,
    capsize=5,
    color="#d2e3fc",
    edgecolor="#1a73e8",
)
ax.set(title="区域满意度均值及标准误", ylabel="满意度（5分制）", ylim=(0, 5))
fig.tight_layout()
plt.show()
